## Imputation of time-series with Darts Nhits


In [ ]:
!nvidia-smi

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import pandas as pd

import sys
sys.path.append("..")
sys.path.append("../models")

from models.darts_wrapper import DartsWrapper
from utils import *

import darts
from darts import TimeSeries

import matplotlib.pyplot as plt


### Load Data

In [ ]:
from dataset_synthetic import get_dataloader

gpu = torch.device("cuda" if torch.cuda.is_available() else "cpu")
train_loader, _, _ = get_dataloader("./data/multivariate_synthetic.csv", device=gpu, batch_size=8)

In [ ]:
main_data = train_loader.dataset.main_data
timepoints = train_loader.dataset.timepoints
total_length = len(main_data)

start = 0
end = total_length - train_loader.dataset.seq_length - train_loader.dataset.valid_length - train_loader.dataset.test_length + 1
train_data = darts.TimeSeries.from_times_and_values(pd.RangeIndex(timepoints[start], timepoints[end]), main_data[start:end])


start = total_length - train_loader.dataset.seq_length - train_loader.dataset.valid_length - train_loader.dataset.test_length + train_loader.dataset.pred_length
end = total_length - train_loader.dataset.seq_length - train_loader.dataset.test_length + train_loader.dataset.pred_length
valid_data = darts.TimeSeries.from_times_and_values(pd.RangeIndex(timepoints[start], timepoints[end]), main_data[start:end])

In [ ]:
print(f"Train data shape: {train_data.shape}")
print(f"Validation data shape: {valid_data.shape}")

## Darts Nhits

In [ ]:
from darts.models import NHiTSModel

darts_model = NHiTSModel(
    input_chunk_length=48,
    output_chunk_length=24,
)

In [ ]:
TRAIN = True

if TRAIN:
    darts_model.fit(train_data,
            val_series=valid_data,
            epochs=100)

In [ ]:
model = DartsWrapper(darts_model, is_probabilistic=False)

In [ ]:
import numpy as np
from collections import defaultdict

horizons = [6, 12, 18]
n_trials = 5

results = {h: defaultdict(list) for h in horizons}

# Run trials
for trial in range(n_trials):
    print(f"Trial {trial+1}/{n_trials}")
    
    for h in horizons:
        out = evaluate_forecasting(
            model,
            test_loader,
            prediction_horizon=h,
            nsample=100,
        )
        
        for key, value in out.items():
            results[h][key].append(value)

# Compute mean & std
print("\n=== Forecasting Results ===")
for h in horizons:
    print(f"\nHorizon: {h}")
    
    for metric in results[h]:
        values = np.array(results[h][metric])
        mean = values.mean()
        std = values.std()
        
        print(f"{metric}: {mean:.4f} ± {std:.4f}")